# 📚 Notebook 1 — Library Management: Class Design

In this notebook we design the **classes** for a small Library Management System.

We're going to walk through **three versions** of the design, from worst to best, so you can *feel* why good OO design matters:

1. ❌ **Bad**: one giant `Library` class that does *everything* (a "god class").
2. ⚠️ **Better**: we split data into separate classes, but responsibilities still leak.
3. ✅ **Best**: each class has one clear job (Single Responsibility Principle) and we apply a few classic OO patterns.

> This notebook focuses on *design*. Notebook 2 turns the best design into a full working system with reservations, search, fines, and notifications.


## 🛠️ Setup

```bash
cd 07-object-oriented-design/library-management
uv sync
```

In VS Code, pick the `.venv` kernel (top-right of the notebook).
If it doesn't appear: `Cmd+Shift+P` → **Reload Window**.


## 📋 Requirements (from the classic interview prompt)

- Members can **search** books by title, author, subject, or publish date.
- Each book has an ISBN; there can be **multiple copies** (we call each copy a *BookItem*).
- Members can **check-out** and **reserve** copies.
- Check-out rules: max **5** active loans per member, loan period **10** days.
- **Fines** apply for late returns (e.g., $0.50/day).
- The system **notifies** members when reserved books become available or when a book is overdue.
- A **librarian** can add/remove books, issue, return, and cancel memberships.

We'll keep the scope small but cover all of these in notebook 2.


## ❌ Version 1 — The "God Class"

A beginner's first attempt is often a single class that holds all the data and all the behaviour.
It *works*, but it's hard to read, test, or extend. Watch the smells.


In [ ]:
from datetime import date, timedelta

class LibraryGod:
    """Anti-pattern: one class does everything."""

    def __init__(self):
        # Parallel lists — easy to get out of sync
        self.book_titles = []       # [title, ...]
        self.book_isbns = []        # [isbn, ...] same index as titles
        self.copies_on_loan = []    # [bool, ...] one per physical copy
        self.copy_isbn = []         # [isbn, ...] which book each copy belongs to
        self.members = []           # [name, ...]
        self.loans = []             # tuples: (member_index, copy_index, due_date)

    def add_book(self, isbn, title):
        self.book_isbns.append(isbn)
        self.book_titles.append(title)

    def add_copy(self, isbn):
        self.copy_isbn.append(isbn)
        self.copies_on_loan.append(False)

    def add_member(self, name):
        self.members.append(name)

    def borrow(self, member_name, isbn, today):
        mi = self.members.index(member_name)
        for ci, (cisbn, out) in enumerate(zip(self.copy_isbn, self.copies_on_loan)):
            if cisbn == isbn and not out:
                self.copies_on_loan[ci] = True
                self.loans.append((mi, ci, today + timedelta(days=10)))
                return ci
        raise RuntimeError("no copies available")

    # ...and on and on: return, fine, search, reserve, notify, print receipts,
    # all living in one class with shared global state.

lib = LibraryGod()
lib.add_book("978-0132350884", "Clean Code")
lib.add_copy("978-0132350884")
lib.add_member("Ada")
copy_id = lib.borrow("Ada", "978-0132350884", date(2026, 4, 1))
print("borrowed copy index:", copy_id)


### 🤔 What's wrong with the god class?

- **Parallel arrays**: `book_titles[i]` and `book_isbns[i]` must stay in sync; one bug and your data is corrupt.
- **No real domain objects**: a "book" isn't a thing, it's scattered state. You can't pass a book around.
- **Every new feature touches this class** (reservations? fines? search?). It becomes thousands of lines.
- **Hard to test**: to test the fine logic you have to set up the whole library.
- **Tight coupling**: borrowing logic knows about list indexes for members *and* copies.

👉 The fix is to give each concept its own class.


## ⚠️ Version 2 — Separate Classes, But Still Messy

We introduce `Book`, `BookItem`, `Member`, and `Library`. Better! But notice how `Library` *still* does too much: it also computes fines and knows about string-formatted receipts.


In [ ]:
from dataclasses import dataclass, field
from datetime import date, timedelta

@dataclass
class Book:
    isbn: str
    title: str

@dataclass
class BookItem:
    barcode: str
    isbn: str
    on_loan: bool = False

@dataclass
class Member:
    id: str
    name: str
    active_loan_barcodes: list = field(default_factory=list)

class LibraryV2:
    def __init__(self):
        self.books = {}       # isbn -> Book
        self.items = {}       # barcode -> BookItem
        self.members = {}     # id -> Member
        self.due = {}         # barcode -> due_date

    def add_book(self, book: Book):
        self.books[book.isbn] = book

    def add_copy(self, isbn: str, barcode: str):
        self.items[barcode] = BookItem(barcode=barcode, isbn=isbn)

    def register(self, m: Member):
        self.members[m.id] = m

    def borrow(self, member_id, isbn, today):
        for item in self.items.values():
            if item.isbn == isbn and not item.on_loan:
                item.on_loan = True
                self.due[item.barcode] = today + timedelta(days=10)
                self.members[member_id].active_loan_barcodes.append(item.barcode)
                # 🚨 Library is also formatting a receipt here — not its job.
                print(f"RECEIPT: {self.members[member_id].name} borrowed {self.books[isbn].title}")
                return item.barcode
        raise RuntimeError("no copies")

    def return_item(self, barcode, today):
        item = self.items[barcode]
        item.on_loan = False
        due = self.due.pop(barcode)
        # 🚨 Fine calculation is hard-coded here too.
        late = max(0, (today - due).days)
        return round(late * 0.5, 2)

lib = LibraryV2()
lib.add_book(Book("978-0132350884", "Clean Code"))
lib.add_copy("978-0132350884", "BK-0001")
lib.register(Member("M-1", "Ada"))
bc = lib.borrow("M-1", "978-0132350884", date(2026, 4, 1))
print("fine:", lib.return_item(bc, date(2026, 4, 20)))


### 🤔 Still imperfect

- `Library` mixes: storage, loan rules, **fine math**, and **receipt printing**.
- Want to add reservations? Another field. Notifications? Another block of prints.
- No clear extension points. Adding a rule = editing `Library` again.

👉 Next version: one class, one job.


## ✅ Version 3 — One Class, One Job (SRP + patterns)

Now we split responsibilities cleanly. Each class has a single, small purpose:

| Class | Responsibility |
|---|---|
| `Book` | Catalog info: ISBN, title, author, subject |
| `BookItem` | A *physical* copy with a barcode and status |
| `Member` | A person who can borrow; tracks their active loans |
| `Loan` | The fact that a member borrowed a BookItem, with due date |
| `Reservation` | A member's claim on a title when no copy is free |
| `Catalog` | **Search** across books (title, author, subject) |
| `FineCalculator` | Converts overdue days to money |
| `LibraryObserver` | Reacts to library events (reservation ready, overdue) — Observer |
| `Library` | **Coordinator** — wires the above together; enforces policies |

### Patterns used (kept tiny and readable)

- **Single Responsibility Principle** — the table above.
- **Strategy** — `FineCalculator` is swappable (e.g., a "student discount" calculator).
- **Observer** — `Library` keeps a *list* of subscribers and publishes events to all of them.

> ⚠️ **Strategy and Observer look alike and are not the same.** Both hand work to an object
> behind an interface, so it is easy to write "Observer" in a design doc and ship a Strategy.
> The distinguishing question is **how many, and who chooses**:
>
> | | Strategy | Observer |
> |---|---|---|
> | How many? | **exactly one**, injected at construction | **zero to many**, added and removed at runtime |
> | Who decides? | the caller picks *the* algorithm | the subscribers opt themselves in |
> | Return value | the subject **uses** it (`fine = calc.fine_for(...)`) | ignored — it is a broadcast |
> | Swapping it | changes *how* the job is done | changes *who else finds out* |
>
> `FineCalculator` is a Strategy: the library needs one number back. `LibraryObserver` is an
> Observer: email, SMS, and an audit log can all listen, and the library does not care.
- **Repository-ish maps** — `Library` owns dictionaries that behave like tiny repositories.

### Class diagram (text form)

```
           ┌───────────────┐          owns
           │    Library    │────────────────────────┐
           └───────────────┘                        ▼
              │       │                  ┌───────────────┐
     manages  │       │ manages          │    Catalog    │  search(title/author/subject)
              ▼       ▼                  └───────────────┘
       ┌────────┐  ┌──────────┐
       │ Member │  │   Book   │ 1 ──── * ┌────────────┐
       └────────┘  └──────────┘          │  BookItem  │
           │ *                           └────────────┘
           │  has active
           ▼
        ┌──────┐        Reservation (member, isbn, created_on)
        │ Loan │ ──┐    Fine        (loan, amount)
        └──────┘   │
                   └── refers to a BookItem

    Library → [LibraryObserver, ...] (Observer): publishes reservation-ready / overdue
    Library → FineCalculator(Strategy): on return
```

We implement this in the next notebook. But first, let's sketch the *skeletons* to cement the idea.

In [ ]:
from dataclasses import dataclass, field
from datetime import date
from typing import Protocol, Optional

# --- Value objects / entities --------------------------------------------

@dataclass(frozen=True)
class Book:
    isbn: str
    title: str
    author: str
    subject: str

@dataclass
class BookItem:
    barcode: str
    book: Book
    on_loan: bool = False

@dataclass
class Member:
    id: str
    name: str
    active_loans: list = field(default_factory=list)  # list[Loan]

@dataclass
class Loan:
    member_id: str
    item: BookItem
    borrowed_on: date
    due_on: date
    returned_on: Optional[date] = None

@dataclass
class Reservation:
    member_id: str
    isbn: str
    created_on: date

# --- Strategy: fine calculation ------------------------------------------

class FineCalculator(Protocol):
    def fine_for(self, loan: Loan, today: date) -> float: ...

class DailyFine:
    def __init__(self, per_day: float = 0.50):
        self.per_day = per_day
    def fine_for(self, loan: Loan, today: date) -> float:
        late = max(0, (today - loan.due_on).days)
        return round(late * self.per_day, 2)

# --- Observer: a SUBSCRIBER LIST, not a single injected collaborator -----

class LibraryObserver(Protocol):
    """Anything that wants to hear about library events implements this.
    Note the `-> None`: an observer's job is to react, not to answer."""
    def on_event(self, event: str, member_id: str, message: str) -> None: ...

class PrintObserver:
    def on_event(self, event: str, member_id: str, message: str) -> None:
        print(f"[{event} -> {member_id}] {message}")

class AuditLog:
    """A SECOND observer. Its existence is the whole point: with a Strategy we
    could only have one of these; with an Observer we can have both at once."""
    def __init__(self): self.entries: list[tuple[str, str]] = []
    def on_event(self, event, member_id, message): self.entries.append((event, member_id))

print("Skeletons OK — full wiring is in notebook 2.")

## 🧭 Recap

- **Bad** design = one god class, parallel arrays, everything coupled.
- **Better** = real classes, but coordinator still does too much.
- **Best** = each class has one job; behaviour that changes independently (fines, notifications) lives behind small interfaces you can swap out.

➡️ Open `02_implementation.ipynb` to run the complete system end-to-end: searching, borrowing, reserving, returning, fining, and notifying.
